# Qwen3 ASR — Transcription and Evaluation

Transcribes `how_irans_flagging_economy_inflamed_its_protests.mp3` using `mlx-community/Qwen3-ASR-1.7B-8bit`, then evaluates the output against a reference transcript using `Qwen2.5-1.5B-Instruct-4bit` on Apple Silicon.

In [ ]:
from pathlib import Path

import mlx.core as mx
from mlx_audio.stt.generate import generate_transcription
from mlx_audio.stt.utils import load_model
from mlx_lm import generate, load

ASR_MODEL = "mlx-community/Qwen3-ASR-1.7B-8bit"
LLM_MODEL = "mlx-community/Qwen2.5-1.5B-Instruct-4bit"

AUDIO = Path("original/audio/how_irans_flagging_economy_inflamed_its_protests.mp3")
TRANSCRIPT_DIR = Path("generated/transcript")
COMPARISON_DIR = Path("generated/comparison")
REFERENCE_TRANSCRIPT = Path("original/transcript") / AUDIO.with_suffix(".md").name

## Transcription

In [ ]:
asr_model = load_model(ASR_MODEL)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
result = generate_transcription(model=asr_model, audio=str(AUDIO))

In [ ]:
TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)

out_file = TRANSCRIPT_DIR / AUDIO.with_suffix(".md").name
out_file.write_text(result.text)
out_file

PosixPath('generated/transcript/how_irans_flagging_economy_inflamed_its_protests.md')

In [ ]:
del asr_model
mx.metal.clear_cache()

mx.metal.clear_cache is deprecated and will be removed in a future version. Use mx.clear_cache instead.


## Transcript Evaluation

Uses `Qwen2.5-1.5B-Instruct-4bit` to evaluate the generated transcript against the reference on three dimensions:
1. Grammatical correctness
2. Completeness
3. Preservation of main points, arguments, and conclusion

In [ ]:
generated_text = out_file.read_text()
reference_text = REFERENCE_TRANSCRIPT.read_text()

In [ ]:
llm, tokenizer = load(LLM_MODEL)

Fetching 9 files:   0%|          | 0/9 [00:00<?, ?it/s]

In [ ]:
prompt_text = f"""You are evaluating an ASR-generated transcript against a reference transcript of the same audio.

## Reference transcript
{reference_text}

## Generated transcript
{generated_text}

Evaluate the generated transcript on the following three dimensions. For each, give a rating (Excellent / Good / Fair / Poor) and a brief explanation with specific examples where relevant.

### 1. Grammatical correctness
Are sentences well-formed? Note any errors, garbled words, or run-ons.

### 2. Completeness
Is all speech from the reference captured? Note any missing sentences, segments, or speakers.

### 3. Preservation of main points, arguments, and conclusion
Are the core ideas, causal arguments, and conclusion retained? Note any omissions or distortions of substance.

### Summary
One paragraph overall assessment.
"""

messages = [{"role": "user", "content": prompt_text}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
comparison = generate(llm, tokenizer, prompt=prompt, max_tokens=1500, verbose=False)

mx.metal.device_info is deprecated and will be removed in a future version. Use mx.device_info instead.


'### Evaluation\n\n**1. Grammatical correctness**\n- **Rating:** Excellent\n- **Explanation:** The generated transcript is well-structured with no grammatical errors. The sentences are coherent and follow proper English grammar rules. For example, the sentence "Inflation in Iran is running at around 50 percent, 5o. That means by the time the bazzaris, the shopkeepers, buy their products and then go to sell them, having to mark up their prices a lot, and then the customers are refusing to pay such high prices." is grammatically correct and clearly conveys the intended meaning.\n\n**2. Completeness**\n- **Rating:** Excellent\n- **Explanation:** The generated transcript captures all speech from the reference without any omissions. The transcript includes all the sentences from the reference, maintaining the original order and structure. For example, the sentence "In late December, a handful of shop owners just outside of the Grand Bazaar had had enough." is included in the generated trans

In [ ]:
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)
comparison_file = COMPARISON_DIR / f"comparison_{AUDIO.stem}.md"
comparison_file.write_text(f"# Transcript Comparison\n\n{comparison}\n")

del llm, tokenizer
mx.metal.clear_cache()